<a href="https://colab.research.google.com/github/siinwook/Dacon/blob/main/%ED%83%80%EC%9D%B4%ED%83%80%EB%8B%89_%EC%83%9D%EC%A1%B4_%EC%98%88%EC%B8%A1_%EC%97%B0%EC%8A%B5%EB%8C%80%ED%9A%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import re as re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score

1. 데이터 불러오기

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dacon/타이타닉 생존 예측 연습대회/train.csv')
test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dacon/타이타닉 생존 예측 연습대회/test.csv')
submission = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dacon/타이타닉 생존 예측 연습대회/submission.csv')

2. EDA

In [ ]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
# SibSp: 함께 탑승한 형제자매, 아내, 남편의 수
# Parch: 함께 탑승한 부모, 자식의 수
# Cabin: 객실번호
# Embarked: 배에 탑승한 위치(C = Cherbourg, Q = Queenstown, S = Southampton)

In [ ]:
train.info()
test.info()

3. 데이터 전처리 함수

In [ ]:
for df in [train, test]:
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+).', expand=False)

    main_titles = ['Mr', 'Miss', 'Mrs', 'Master']
    df.loc[~df['Title'].isin(main_titles), 'Title'] = 'Others'

    title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Others": 5}
    df['Title'] = df['Title'].map(title_mapping)

    df['Title'] = df['Title'].fillna(1)

In [ ]:
# Title: {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Others": 5}

In [ ]:
for df in [train, test]:
  df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

In [ ]:
# FaimilySize: 가족 수

In [ ]:
def preprocess(df):
  # 결측치 처리
  df['Age'] = df['Age'].fillna(df['Age'].mean())
  df['Fare'] = df['Fare'].fillna(df['Fare'].mean())

  # 인코딩
  df['Sex'] = df['Sex'].map({'male' : 0, 'female' : 1})

  df['Embarked'] = df['Embarked'].map({'C' : 0, 'Q': 1, 'S':2})
  df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mean())

  df['Cabin'] = df['Cabin'].str[:1]
  df['Cabin'] = df['Cabin'].map({'C' : 0, 'B': 1, 'D':2, 'E':3, 'A':4,'F':5,'G':6,'T':7})
  df['Cabin'] = df['Cabin'].fillna(df['Cabin'].mean())

  # Sex_Pclass 새로운 변수 생성
  df['Sex_Pclass'] = df['Sex'].astype(str) + "-" + df['Pclass'].astype(str)
  Sex_Plcass_mapping = {"0-1": 1, "0-2": 2, "0-3": 3, "1": 4, "1-2": 5, "1-3": 6}
  df['Sex_Pclass'] = df['Sex_Pclass'].map(Sex_Plcass_mapping)
  df['Sex_Pclass'] = df['Sex_Pclass'].fillna(df['Sex_Pclass'].mean())

  features = ['Pclass', 'Sex', 'Age','Fare', 'Cabin',	'Embarked', 'Title', 'FamilySize','Sex_Pclass']

  return df[features]

In [ ]:
xtrain=preprocess(train)
ytrain=train['Survived']
xtest=preprocess(test)
ytest=submission['Survived']

4. train_test_split

In [ ]:
# train, test 셋이 이미 나눠짐

5. 모델 생성

In [ ]:
params = {'max_depth' : [3,4,5,6,7,8,9,10,None], 'min_samples_split' : [2, 3, 5, 7, 11], 'min_samples_leaf' : [1,2,4] }
model = GridSearchCV(RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    random_state=0
),estimators= params, cv=5)
model.fit(xtrain,ytrain)
model.best_params_

{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 7}

In [ ]:
#params = {'C' : 10**np.linspace(-5,5,21)}
#model=GridSearchCV(LogisticRegression(random_state=0,max_iter=100000),params,cv=5)
#model.fit(xtrain,ytrain)
#model.best_params_

6. 성능확인

In [ ]:
best_model = model.best_estimator_ # 최고 모델만 남겨서 제출할 때 용이
print(best_model.score(xtrain,ytrain))
print(best_model.score(xtest,ytest))
ypred_proba=best_model.predict_proba(xtest)[:,1]
ypred=best_model.predict(xtest)

0.9023569023569024
0.8899521531100478


7. 제출


In [ ]:
submission['Survived'] = ypred_proba

filename = 'submission.csv'
submission.to_csv(filename,index=False)